# 🎯 FIXED: Deepfake Training (Auto-detects paths)

## Changes:
- Auto-detects Kaggle dataset structure
- Validates files before training
- Better error messages

---
**SETUP:** Runtime → Change runtime type → GPU (T4)

In [ ]:
# GPU Check
import tensorflow as tf
print('GPU:', tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    raise SystemExit('❌ Enable GPU!')
print('✅ GPU Ready\n')

In [ ]:
# Dependencies & Kaggle
!pip install -q kaggle opencv-python pillow requests

import json, os, shutil
!mkdir -p ~/.kaggle
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": "snapdragoon77", "key": "KGAT_a0e47466be9a5de9ac5b0e203f5e42c0"}, f)
!chmod 600 ~/.kaggle/kaggle.json
print('✅ Kaggle configured')

In [ ]:
# Config
TARGET_SIZE = (299, 299)
BATCH_SIZE = 32
FROZEN_EPOCHS = 10
FINETUNE_EPOCHS = 15

WORKSPACE = '/content/deepfake_training'
RAW_DIR = f'{WORKSPACE}/raw'
PROCESSED_DIR = f'{WORKSPACE}/processed'

if os.path.exists(WORKSPACE):
    shutil.rmtree(WORKSPACE)

for p in [f'{RAW_DIR}/real', f'{RAW_DIR}/fake', f'{PROCESSED_DIR}/real', f'{PROCESSED_DIR}/fake']:
    os.makedirs(p, exist_ok=True)

print('✅ Workspace ready')

In [ ]:
# FIXED: Download & Extract Kaggle Dataset
print('Downloading Kaggle dataset...')
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces

print('\nExtracting...')
!unzip -q 140k-real-and-fake-faces.zip -d /content/kaggle_data/

print('\nSearching for images...')
# Auto-detect structure
import glob
from pathlib import Path

# Find all directories with images
all_dirs = !find /content/kaggle_data -type d
all_dirs = [d for d in all_dirs if d]

print(f'Found {len(all_dirs)} directories')
for d in all_dirs[:10]:  # Show first 10
    img_count = len([f for f in os.listdir(d) if f.endswith(('.jpg', '.png'))])
    if img_count > 0:
        print(f'  {d}: {img_count} images')

# Try common paths
possible_paths = [
    '/content/kaggle_data/real_vs_fake/real',
    '/content/kaggle_data/real_vs_fake/fake',
    '/content/kaggle_data/real',
    '/content/kaggle_data/fake',
    '/content/kaggle_data/train/real',
    '/content/kaggle_data/train/fake',
]

real_source = None
fake_source = None

for path in possible_paths:
    if os.path.exists(path):
        if 'real' in path.lower():
            real_source = path
        elif 'fake' in path.lower():
            fake_source = path

print(f'\nDetected paths:')
print(f'  REAL: {real_source}')
print(f'  FAKE: {fake_source}')

# Copy files
if real_source and os.path.exists(real_source):
    !cp {real_source}/*.jpg {RAW_DIR}/real/ 2>/dev/null || true
    !cp {real_source}/*.png {RAW_DIR}/real/ 2>/dev/null || true

if fake_source and os.path.exists(fake_source):
    !cp {fake_source}/*.jpg {RAW_DIR}/fake/ 2>/dev/null || true
    !cp {fake_source}/*.png {RAW_DIR}/fake/ 2>/dev/null || true

real_count = len([f for f in os.listdir(f'{RAW_DIR}/real') if f.endswith(('.jpg', '.png'))])
fake_count = len([f for f in os.listdir(f'{RAW_DIR}/fake') if f.endswith(('.jpg', '.png'))])

print(f'\n✅ Copied: {real_count:,} real, {fake_count:,} fake')

if real_count == 0:
    print('\n⚠️ WARNING: No REAL images found!')
    print('Checking all subdirectories...')
    !find /content/kaggle_data -name '*.jpg' -type f | head -20

In [ ]:
# Download GAN faces
import requests, time

print('Downloading GAN faces...')
url = 'https://thispersondoesnotexist.com/'
headers = {'User-Agent': 'Mozilla/5.0'}

for i in range(2000):
    try:
        r = requests.get(url, headers=headers, timeout=10)
        if r.status_code == 200:
            with open(f'{RAW_DIR}/fake/gan_{i:04d}.jpg', 'wb') as f:
                f.write(r.content)
        if (i + 1) % 200 == 0:
            print(f'   {i+1}/2000')
        time.sleep(0.3)
    except:
        continue

gan_count = len([f for f in os.listdir(f'{RAW_DIR}/fake') if f.startswith('gan_')])
print(f'✅ GAN: {gan_count}')

# Final count
real_total = len([f for f in os.listdir(f'{RAW_DIR}/real') if f.endswith(('.jpg', '.png'))])
fake_total = len([f for f in os.listdir(f'{RAW_DIR}/fake') if f.endswith(('.jpg', '.png'))])

print(f'\n=== DOWNLOAD SUMMARY ===')
print(f'REAL: {real_total:,}')
print(f'FAKE: {fake_total:,}')
print(f'TOTAL: {real_total + fake_total:,}\n')

if real_total == 0:
    raise SystemExit('❌ ERROR: No REAL images! Check Kaggle download.')

In [ ]:
# Preprocessing
import cv2, numpy as np

print('=== PREPROCESSING ===')
cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

def process_face(img_path, out_path, size=(299, 299)):
    try:
        img = cv2.imread(img_path)
        if img is None:
            return False
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        faces = cascade.detectMultiScale(gray, 1.1, 4, minSize=(80, 80))
        if len(faces) == 0:
            return False
        x, y, w, h = max(faces, key=lambda r: r[2] * r[3])
        margin = int(w * 0.2)
        x, y = max(0, x - margin), max(0, y - margin)
        w = min(img.shape[1] - x, w + 2 * margin)
        h = min(img.shape[0] - y, h + 2 * margin)
        face = cv2.resize(img[y:y+h, x:x+w], size)
        cv2.imwrite(out_path, face)
        return True
    except:
        return False

for label in ['real', 'fake']:
    src = f'{RAW_DIR}/{label}'
    dst = f'{PROCESSED_DIR}/{label}'
    files = [f for f in os.listdir(src) if f.endswith(('.jpg', '.png'))][:4000]
    print(f'\n{label.upper()}: {len(files)} images')
    saved = 0
    for i, f in enumerate(files):
        if process_face(os.path.join(src, f), os.path.join(dst, f), TARGET_SIZE):
            saved += 1
        if (i + 1) % 500 == 0:
            print(f'   {i+1}/{len(files)} - {saved} saved')
    print(f'✅ {label}: {saved} faces')

print('\n✅ Preprocessing done')

In [ ]:
# Balance dataset
real_files = [f for f in os.listdir(f'{PROCESSED_DIR}/real') if f.endswith(('.jpg', '.png'))]
fake_files = [f for f in os.listdir(f'{PROCESSED_DIR}/fake') if f.endswith(('.jpg', '.png'))]

print(f'Before: REAL={len(real_files):,}, FAKE={len(fake_files):,}')

target = min(len(real_files), len(fake_files), 3500)

for label, files in [('real', real_files), ('fake', fake_files)]:
    if len(files) > target:
        for f in files[target:]:
            os.remove(f'{PROCESSED_DIR}/{label}/{f}')

print(f'After: REAL={target:,}, FAKE={target:,}, TOTAL={target*2:,} ✅')

In [ ]:
# Load data & build model
from tensorflow import keras
from keras import mixed_precision
from keras.applications import Xception
from keras.layers import Dense, GlobalAveragePooling2D, Dropout
from keras.models import Model
from keras.optimizers import Adam

train_ds = keras.utils.image_dataset_from_directory(
    PROCESSED_DIR, validation_split=0.2, subset='training', seed=123,
    image_size=TARGET_SIZE, batch_size=BATCH_SIZE, label_mode='categorical'
)

val_ds = keras.utils.image_dataset_from_directory(
    PROCESSED_DIR, validation_split=0.2, subset='validation', seed=123,
    image_size=TARGET_SIZE, batch_size=BATCH_SIZE, label_mode='categorical'
)

def preprocess(images, labels):
    return keras.applications.xception.preprocess_input(images), labels

train_ds = train_ds.map(preprocess).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess).prefetch(tf.data.AUTOTUNE)

mixed_precision.set_global_policy('mixed_float16')

with tf.device('/GPU:0'):
    base = Xception(weights='imagenet', include_top=False, input_shape=(299, 299, 3))
    base.trainable = False
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation='relu', dtype='float32')(x)
    x = Dropout(0.5)(x)
    out = Dense(2, activation='softmax', dtype='float32')(x)
    model = Model(base.input, out)

print(f'✅ Model: {model.count_params():,} params')

In [ ]:
# Phase 1: Frozen base
from keras.callbacks import EarlyStopping, ReduceLROnPlateau

print('=== PHASE 1: FROZEN ===')
model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

h1 = model.fit(train_ds, epochs=FROZEN_EPOCHS, validation_data=val_ds, callbacks=[
    EarlyStopping('val_loss', patience=3, restore_best_weights=True),
    ReduceLROnPlateau('val_loss', factor=0.5, patience=2)
])

print(f'✅ Phase 1: {max(h1.history["val_accuracy"]):.1%}')

In [ ]:
# Phase 2: Fine-tune
print('=== PHASE 2: FINE-TUNE ===')
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

h2 = model.fit(train_ds, epochs=FINETUNE_EPOCHS, validation_data=val_ds, callbacks=[
    EarlyStopping('val_loss', patience=2, restore_best_weights=True),
    ReduceLROnPlateau('val_loss', factor=0.3, patience=2)
])

final_acc = max(h2.history['val_accuracy'])
print(f'✅ Phase 2: {final_acc:.1%}')

In [ ]:
# Export TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

with open('deepfake_net.tflite', 'wb') as f:
    f.write(tflite_model)

size_mb = len(tflite_model) / 1024 / 1024
print(f'✅ Exported: deepfake_net.tflite ({size_mb:.1f} MB)')

# Test
interpreter = tf.lite.Interpreter(model_path='deepfake_net.tflite')
interpreter.allocate_tensors()
test_img = np.random.rand(1, 299, 299, 3).astype(np.float32)
test_img = (test_img * 255 - 127.5) / 127.5
interpreter.set_tensor(interpreter.get_input_details()[0]['index'], test_img)
interpreter.invoke()
output = interpreter.get_tensor(interpreter.get_output_details()[0]['index'])
print(f'✅ Test passed: {output[0]}')

In [ ]:
# Summary
print('\n' + '='*70)
print('🎉 TRAINING COMPLETE!')
print('='*70)
print(f'🎯 Accuracy: {final_acc:.1%}')
print(f'📦 Model: deepfake_net.tflite ({size_mb:.1f} MB)')
print(f'📥 Download from Files panel → deepfake_net.tflite')
print('='*70)